### Agent Tool 실행 검증과 통제

Agent는 사용자 요청에 따라 실행할 Tool과 인자를 선택한다. Tool을 실행하는 프로그램은 인자, 사용자 권한, 승인 여부를 확인하고 허용된 작업만 실행해야 한다.


In [2]:
import json
from typing import Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ToolCallLimitMiddleware,
    ToolCallRequest,
    wrap_model_call,
    wrap_tool_call,
)
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv()
MODEL_NAME = "gemini-3.6-flash"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

### 계좌 조회와 송금 예제

사용자가 잔액 조회와 1만 원 송금을 요청하는 상황을 살펴보자. 조회는 접근 가능한 계좌에만 허용하고, 송금은 금액과 권한을 확인한 뒤 승인을 받아 실행한다.

먼저 실행 전 검사를 적용하고 LLM의 호출이 거절되는 흐름을 확인한다. 이어서 조회 결과 검사를 추가하고, 마지막으로 호출 횟수를 제한한다.

### Middleware
Middleware는 모델이나 Tool 호출 전후에 공통 처리를 적용하는 기능이다. `create_agent`에 등록하면 Agent가 실행 과정에서 해당 Middleware를 호출한다.

```text
agent 시작
   │
   ▼
before_agent
   │
   ▼
before_model
   │
   ▼
┌─────────────────┐
│   model call    │  ← wrap_model_call
└─────────────────┘
   │
   ▼
after_model
   │
   ├── tool call 있음
   │       │
   │       ▼
   │   tool execution  ← wrap_tool_call
   │       │
   │       └──────────→ 다시 before_model
   │
   └── tool call 없음
           │
           ▼
      after_agent
```

일반적인 Agent 실행 흐름이다. `before_agent`와 `after_agent`는 Agent 실행의 시작과 종료 시점에 호출되고, 모델·Tool 관련 Middleware는 반복 과정에서 호출된다. `wrap_model_call`과 `wrap_tool_call`은 해당 호출을 감싸 실행 전후를 처리하거나 실행을 차단할 수 있다. Tool 호출이 여러 개면 `wrap_tool_call`은 각 호출에 적용된다.

`@wrap_tool_call`은 Tool 호출을 감싸 실행 전후를 처리하는 Middleware를 만든다. LangChain은 이 함수에 `request`와 `handler`를 전달한다. `request`는 현재 호출 정보를 담은 객체로, 모델 Middleware에서는 모델 호출 정보, Tool Middleware에서는 Tool 호출 정보를 받는다. Tool의 호출 이름과 인자는 `request.tool_call`에서 확인한다.

`handler`는 전달받은 요청으로 다음 처리를 진행하는 함수다. Tool Middleware에서 `handler(request)`를 호출하면 다음 Middleware 또는 실제 Tool 실행으로 이어진다. 호출 전에는 인자·권한·승인을 검사하고, 반환 후에는 실행 결과를 검사할 수 있다. 실행을 차단하려면 `handler`를 호출하지 않고 사유를 담은 `ToolMessage`를 반환한다. 단, 실행 후 검사는 이미 수행된 작업을 되돌리지 못한다.

### Tool 인자 검증하기

계좌 조회에 필요한 계좌 ID와 송금에 필요한 금액이 올바른지 검사한다.

In [10]:
class LookupAccountArgs(BaseModel):
    account_id: str = Field(min_length=3, max_length=40)


class TransferArgs(BaseModel):
    account_id: str = Field(min_length=3, max_length=40)
    amount: int = Field(gt=0, le=1000000)
    request_id: str = Field(min_length=8, max_length=80)

### 실행 권한과 승인 확인하기

인자 형식이 올바르더라도 접근 권한이 없거나 승인되지 않은 송금은 실행하지 않아야 한다.

`validate_proposal`은 Tool 이름, 인자 형식, 계좌 접근 권한, 송금 한도, 승인 여부를 검사한다. 검사를 통과하면 검증된 인자를 반환하고, 통과하지 못하면 처리 상태와 사유를 딕셔너리로 반환한다.

In [11]:
TOOL_SCHEMAS = {'lookup_account': LookupAccountArgs, 'transfer': TransferArgs}

# 실제 송금에 적용할 업무 한도는 10만 원이다.
MAX_TRANSFER = 100000


def failure(code: str, message: str) -> dict:
    return {'status': 'failed', 'code': code, 'message': message, 'data': {}}


def validate_proposal(proposal: dict, context: dict):
    # 사용자가 취소한 요청은 검사와 실행을 중단한다.
    if context['cancelled']:
        return {'status': 'cancelled', 'code': 'USER_CANCELLED', 'message': '요청이 취소되었습니다.', 'data': {}}

    # Tool 이름에 맞는 검증 모델을 찾고, 등록되지 않은 이름은 거절한다.
    schema = TOOL_SCHEMAS.get(proposal['name'])
    if schema is None:
        return failure('UNKNOWN_TOOL', '허용되지 않은 Tool입니다.')

    # 선택한 Tool의 모델로 필수 인자와 값의 범위를 검사한다.
    try:
        args = schema.model_validate(proposal['args'])
    except ValidationError:
        return failure('INVALID_ARGUMENT', 'Tool 인자가 올바르지 않습니다.')

    # 사용자에게 접근 권한이 있는 계좌인지 확인한다.
    if args.account_id not in context['allowed_accounts']:
        return failure('FORBIDDEN', '이 계좌에 접근할 권한이 없습니다.')

    # 송금에는 업무 한도와 승인 검사를 추가로 적용한다.
    if proposal['name'] == 'transfer':
        # 인자 형식을 통과해도 업무 한도를 넘으면 거절한다.
        if args.amount > MAX_TRANSFER:
            return failure('LIMIT_EXCEEDED', '송금 한도를 초과했습니다.')

        # 승인이 거절된 요청은 취소 상태로 종료한다.
        if args.request_id in context['rejected_request_ids']:
            return {'status': 'cancelled', 'code': 'APPROVAL_REJECTED', 'message': '승인이 거절되었습니다.', 'data': {}}

        # 승인 기록이 없으면 실행하지 않고 승인을 기다린다.
        if args.request_id not in context['approved_request_ids']:
            return {'status': 'waiting_for_human', 'code': 'APPROVAL_REQUIRED', 'message': '송금 승인을 기다립니다.', 'data': {'request_id': args.request_id, 'amount': args.amount}}

    # 모든 검사를 통과한 Pydantic 인자 객체를 반환한다.
    return args

### Tool과 모델 준비하기

조회와 송금 Tool을 `@tool`로 정의한다. 조회 Tool은 계좌 ID, 잔액, 통화가 담긴 예제 데이터를 반환한다. 송금 Tool의 본문은 `pass`로 두고 승인에 따른 실행 여부만 확인한다.

In [12]:

account_responses = {
    "acct-001": {
        "account_id": "acct-001",
        "balance": 80000,
        "currency": "KRW",
    },
}


@tool("lookup_account", args_schema=LookupAccountArgs)
def lookup_account_tool(account_id: str):
    """계좌 ID로 잔액을 조회한다."""
    return account_responses[account_id].copy()


@tool("transfer", args_schema=TransferArgs)
def transfer_tool(account_id: str, amount: int, request_id: str):
    """계좌 ID, 금액, 요청 ID로 송금을 요청한다."""
    pass


tools = [lookup_account_tool, transfer_tool]

### 첫 요청에서 Tool 호출 제안하기

이 예제에서는 첫 모델 호출에 `tool_choice="any"`를 적용해 Tool 호출을 생성하게 한다. Tool 결과를 받은 뒤에는 `"none"`으로 추가 호출 없이 안내한다.

`@wrap_model_call`은 모델 호출을 감싸는 Middleware를 만든다. 아래에서는 `request.override(tool_choice=...)`로 모델 호출 설정을 변경한 요청을 만들고, `handler`에 전달해 해당 설정으로 모델을 호출한다.

In [5]:
@wrap_model_call
def force_tool_call(request, handler):
    # 첫 요청에서 Tool 호출을 강제하고, 결과를 받은 뒤에는 안내만 생성한다.
    if isinstance(request.messages[-1], ToolMessage):
        tool_choice = "none"
    else:
        tool_choice = "any"

    return handler(request.override(tool_choice=tool_choice))

### 실행 Context 전달하기

State는 Agent가 작업하면서 갱신하는 정보이고, Context는 프로그램이 Agent에 전달하는 실행 기준이다. 예를 들어 대화 기록은 State에, 사용자 권한은 Context에 담는다.

| 구분 | 전달하는 주체 | 예 |
|---|---|---|
| Tool 인자 | LLM이 호출을 제안할 때 생성 | 조회할 계좌 ID, 송금할 금액 |
| 실행 Context | 프로그램이 Agent 실행 시 전달 | 접근 가능한 계좌, 승인된 요청 ID |

LLM이 어떤 계좌를 조회하자고 제안하더라도, 실행 여부는 프로그램이 전달한 권한 정보로 판단한다.

```text
agent.invoke(..., context=execution_context)  → 실행할 때 실제 값 전달
request.runtime.context  → Middleware에서 전달받은 값 사용
```

아래에서 `execution_context`를 준비하고, Agent를 실행할 때 `context` 인자로 전달한다.

### 실행 전 검증 Middleware 만들기

`validate_tool_call`은 Tool 호출을 감싸 실행 여부를 제어한다. `request.tool_call`의 호출 정보와 `request.runtime.context`의 권한 정보를 `validate_proposal`에 전달한다.

이름·인자·권한·승인 검사를 통과한 경우에만 `handler(request)`를 호출해 다음 Middleware 또는 실제 Tool 실행으로 이어간다. 거절되면 Tool 실행 없이 사유를 반환한다.

In [14]:
execution_context = {
    "user_id": "user-1",
    "allowed_accounts": {"acct-001"},
    "approved_request_ids": set(),
    "rejected_request_ids": set(),
    "cancelled": False,
}


@wrap_tool_call
def validate_tool_call(request: ToolCallRequest, handler):
    print("실행 전 검사:", request.tool_call["name"])
    result = validate_proposal(request.tool_call, request.runtime.context)

    # 거절·취소·승인 대기이면 Tool 실행 없이 사유를 반환한다.
    if isinstance(result, dict):
        return ToolMessage(
            content=json.dumps(result, ensure_ascii=False),
            tool_call_id=request.tool_call["id"],
            name=request.tool_call["name"],
            status="error",
        )

    print("Tool 실행:", request.tool_call["name"])
    return handler(request)

### Agent에 실행 전 검사 적용하기

`create_agent`의 `middleware`에 모델 설정과 실행 전 검사를 등록한다. Agent가 Tool을 호출할 때 등록된 검사가 실행된다.

In [ ]:
system_prompt = (
    "사용자가 지정한 계좌 ID와 금액으로 Tool 호출을 제안하세요. "
    "Tool 결과에 근거해 한국어로 안내하세요. "
    "오류가 있으면 사유를 안내하고 잔액이나 성공 여부를 추측하지 마세요."
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    middleware=[force_tool_call, validate_tool_call],
)

### 권한이 없는 계좌 조회하기

`acct-999` 조회를 요청한다. 실행 전 검사에서 `FORBIDDEN`이 반환되고, `Tool 실행`은 출력되지 않는지 확인한다. 마지막에는 LLM이 거절 사유를 안내한다.

In [24]:

message = { 
    '성공' : "acct-001 계좌의 잔액을 조회해 줘.",
    '실패-1' : "acct-991 계좌의 잔액을 조회해 줘.",
    '실패-2' : "a1 계좌의 잔액을 조회해 줘.",
    }

result = agent.invoke(
    {"messages": [HumanMessage(content=message['실패-2'])]},
    context=execution_context,
)

for message in result["messages"]:
    message.pretty_print()

실행 전 검사: lookup_account
================================ Human Message =================================

a1 계좌의 잔액을 조회해 줘.
================================== Ai Message ==================================

[]
Tool Calls:
  lookup_account (call_1027626)
 Call ID: call_1027626
  Args:
    account_id: a1
================================= Tool Message =================================
Name: lookup_account

{"status": "failed", "code": "INVALID_ARGUMENT", "message": "Tool 인자가 올바르지 않습니다.", "data": {}}
================================== Ai Message ==================================

[{'type': 'text', 'text': 'a1 계좌의 잔액을 조회하는 중 오류가 발생했습니다.\n\n* **오류 사유:** Tool 인자가 올바르지 않습니다. (INVALID_ARGUMENT)\n\n계좌 잔액을 확인할 수 없습니다.', 'extras': {'signature': 'EvMDCvADAWkUfRMPadFbjM9ouUlJKs7JmFR4DLZkhCjhD+hD+Ojg5j+XPCznJ8eWbMURtJA2wQDarcxKaXbZCCEWimtZn+EZaxFrSTzfzxwzNHrTBXW5rvcMYa2zJ+HU6JGx16bJ2rEiNN73q9h6Dp7EUfxkUaiy7GehZMdxUVuLpUaapUt2ISeTmJoxR8lL04/MSdYlp72cX9AXKHBJvNEpE42zHRWTOn9o9VYSnPxgrzW19+9Q1M8y9frZgGQd

### 다른 거절 조건 확인하기

각 거절 조건을 확인하기 위해 Tool 호출 정보를 직접 만들어 검증 함수에 전달한다. 미등록 Tool, 잘못된 인자, 접근 권한 없음, 송금 한도 초과, 승인 대기 결과를 확인한다.

In [19]:
# 접근 가능한 계좌를 지정하고, 승인·거절·취소 정보는 없는 상태로 시작한다.
context = {'user_id': 'user-1', 'allowed_accounts': {'acct-001'}, 'approved_request_ids': set(), 'rejected_request_ids': set(), 'cancelled': False}

cases = [
    # 미등록 Tool: 허용 목록에 없는 Tool을 요청한다.
    {
        'name': 'delete_account',
        'args': {'account_id': 'acct-001'},
    },
    # 잘못된 인자: 계좌 ID가 최소 길이보다 짧다.
    {
        'name': 'lookup_account',
        'args': {'account_id': 'x'},
    },
    # 권한 없음: 접근할 수 없는 계좌를 조회한다.
    {
        'name': 'lookup_account',
        'args': {'account_id': 'acct-999'},
    },
    # 한도 초과: 업무 한도인 10만 원을 넘겨 송금한다.
    {
        'name': 'transfer',
        'args': {
            'account_id': 'acct-001',
            'amount': 150000,
            'request_id': 'request-01',
        },
    },
    # 승인 정보 위조: 인자에 approved=True를 넣어도 실제 승인으로 인정하지 않는다.
    {
        'name': 'transfer',
        'args': {
            'account_id': 'acct-001',
            'amount': 10000,
            'request_id': 'request-02',
            'approved': True,
        },
    },
]
for proposal in cases:
    print(proposal['name'], validate_proposal(proposal, context))

delete_account {'status': 'failed', 'code': 'UNKNOWN_TOOL', 'message': '허용되지 않은 Tool입니다.', 'data': {}}
lookup_account {'status': 'failed', 'code': 'INVALID_ARGUMENT', 'message': 'Tool 인자가 올바르지 않습니다.', 'data': {}}
lookup_account {'status': 'failed', 'code': 'FORBIDDEN', 'message': '이 계좌에 접근할 권한이 없습니다.', 'data': {}}
transfer {'status': 'failed', 'code': 'LIMIT_EXCEEDED', 'message': '송금 한도를 초과했습니다.', 'data': {}}
transfer {'status': 'waiting_for_human', 'code': 'APPROVAL_REQUIRED', 'message': '송금 승인을 기다립니다.', 'data': {'request_id': 'request-02', 'amount': 10000}}


### 실행 후 결과 검증 추가하기

결과 검증은 외부 Tool이나 API처럼 반환값을 보장하기 어려운 경우에 유용하다. 직접 만든 Tool도 외부 데이터를 사용하면 검증이 필요할 수 있다. 이 예제에서는 잘못된 조회 결과를 넣어 검증 흐름을 확인한다.

실행 전 검사는 LLM이 제안한 호출을 대상으로 한다. 실행 후 검사는 Tool이 반환한 데이터를 대상으로 한다. 조회 결과에 계좌 ID, 잔액, 통화가 있는지 검사하고 요청한 계좌와 같은지 확인한다.

결과 검증 Middleware에서는 먼저 `handler(request)`로 Tool을 실행한 뒤, 반환된 `ToolMessage`를 검사한다. 잘못된 결과는 LLM에 전달되기 전에 오류 메시지로 바꾼다.

In [ ]:
class LookupResponse(BaseModel):
    account_id: str
    balance: int
    currency: Literal["KRW"]

### 결과 검증 Middleware 만들기

`validate_tool_result`는 조회 결과만 검사한다. 필수 필드나 형식이 잘못되면 `INVALID_TOOL_RESULT`, 계좌가 다르면 `RESULT_MISMATCH`를 반환한다. 정상 결과와 이미 실패한 Tool 결과는 그대로 전달한다.

In [ ]:
@wrap_tool_call
def validate_tool_result(request: ToolCallRequest, handler):
    message = handler(request)

    # 조회 결과만 검사하고, 이미 실패한 결과는 그대로 전달한다.
    if request.tool_call["name"] != "lookup_account" or message.status == "error":
        return message

    print("실행 후 검사:", request.tool_call["name"])
    error = None
    try:
        response = LookupResponse.model_validate_json(message.content)
    except ValidationError:
        error = failure("INVALID_TOOL_RESULT", "조회 결과의 필드나 형식이 올바르지 않습니다.")
    else:
        if response.account_id != request.tool_call["args"]["account_id"]:
            error = failure("RESULT_MISMATCH", "요청한 계좌와 반환된 계좌가 다릅니다.")

    # 잘못된 결과 대신 오류 사유를 LLM에 전달한다.
    if error is not None:
        return ToolMessage(
            content=json.dumps(error, ensure_ascii=False),
            tool_call_id=request.tool_call["id"],
            name=request.tool_call["name"],
            status="error",
        )

    return message

### 결과 검사까지 적용하기

실행 전 검사 뒤에 결과 검증 Middleware를 등록한다. 호출은 `validate_tool_call → validate_tool_result → Tool` 순서로 들어가고, Tool 결과가 돌아올 때 `validate_tool_result`가 반환값을 검사한다.

실행 전 검사에서 거절하면 `handler`를 호출하지 않으므로 Tool 실행과 결과 검사도 진행되지 않는다.

In [ ]:
result_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    middleware=[force_tool_call, validate_tool_call, validate_tool_result],
)

### 조회 결과에 따라 응답 확인하기

접근 가능한 `acct-001`을 조회한다. `result_case`를 `"정상"`, `"필드 누락"`, `"계좌 불일치"`로 바꾸어 실행한다.

실행 전 검사, Tool 실행, 실행 후 검사가 순서대로 출력되는지 확인한다. 잘못된 조회 결과는 오류로 바뀌고, 마지막 LLM 응답에서도 잘못된 잔액을 안내하지 않아야 한다.

In [ ]:
result_cases = {
    "정상": {"account_id": "acct-001", "balance": 80000, "currency": "KRW"},
    "필드 누락": {"account_id": "acct-001", "currency": "KRW"},
    "계좌 불일치": {"account_id": "acct-999", "balance": 80000, "currency": "KRW"},
}
result_case = "정상"

# Tool이 반환할 데이터를 바꾸어 결과 검사를 확인한다.
original_response = account_responses["acct-001"]
account_responses["acct-001"] = result_cases[result_case]
try:
    result = result_agent.invoke(
        {"messages": [HumanMessage(content="acct-001 계좌의 잔액을 조회해 줘.")]},
        context=execution_context,
    )
    for message in result["messages"]:
        message.pretty_print()
finally:
    account_responses["acct-001"] = original_response

### 호출 횟수 제한 추가하기

`ToolCallLimitMiddleware`는 Tool 호출 횟수를 관리하는 내장 Middleware다. `run_limit`으로 한 번의 Agent 실행에서 허용할 호출 수를 지정한다.

한도를 넘긴 호출에는 오류 메시지를 반환한다. `exit_behavior="continue"`이면 모델이 그 결과를 받아 사용자에게 안내할 수 있다.

In [ ]:
limited_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    middleware=[
        ToolCallLimitMiddleware(run_limit=0, exit_behavior="continue"),
        force_tool_call,
        validate_tool_call,
        validate_tool_result,
    ],
)

### 호출 한도로 실행 차단하기

접근 가능한 계좌를 조회하되 호출 한도를 `0`으로 설정한다. 반환된 `ToolMessage`에 한도 초과 안내가 있고 Tool은 실행되지 않는지 확인한다.

위 셀의 `run_limit`을 `1`로 바꾸어 Agent를 다시 만들면 한 번의 조회가 허용된다. 호출 횟수는 새 실행마다 초기화된다.

In [ ]:
result = limited_agent.invoke(
    {"messages": [HumanMessage(content="acct-001 계좌의 잔액을 조회해 줘.")]},
    context=execution_context,
)

for message in result["messages"]:
    message.pretty_print()

### 특정 Tool의 호출 횟수 제한하기

`tool_name`을 지정하면 해당 Tool에만 한도를 적용한다. 다음 설정을 `middleware`에 등록하면 `lookup_account` 호출을 한 번의 Agent 실행에서 최대 2회로 제한한다.

```python
ToolCallLimitMiddleware(
    tool_name="lookup_account",
    run_limit=2,
)
```

### 검사 위치 정리

| 검사 | 실행 시점 | 적용 방식 |
|---|---|---|
| 인자·권한·승인 | Tool 실행 전 | `@wrap_tool_call`에서 `handler(request)` 호출 전에 검사 |
| 반환값 | Tool 실행 후, LLM 전달 전 | `@wrap_tool_call`에서 `handler(request)`가 반환한 결과 검사 |
| 호출 횟수 | 허용 횟수를 넘긴 호출의 실행 전 | `ToolCallLimitMiddleware` |

검증 함수는 검사 조건을 정의하고, Middleware는 그 검사를 실행할 시점에 연결한다.

### Tool이 많아질 때

Tool마다 필요한 검증은 다르다. 현재 예제는 `validate_proposal` 안에서 Tool 이름으로 구분하지만, Tool이 많아지면 검증 함수를 각각 정의하고 이름으로 연결할 수 있다.

다음은 `validate_lookup`과 `validate_transfer`를 별도로 정의했을 때의 연결 예시다. 각 함수는 Tool 인자와 Context를 받아, 기존 `validate_proposal`처럼 통과하면 검증된 인자 객체를, 거절하면 사유를 담은 딕셔너리를 반환한다고 가정한다.

```python
TOOL_GUARDS = {
    "lookup_account": validate_lookup,
    "transfer": validate_transfer,
}


@wrap_tool_call
def validate_by_tool(request: ToolCallRequest, handler):
    tool_call = request.tool_call
    guard = TOOL_GUARDS.get(tool_call["name"])

    if guard is None:
        result = failure("UNKNOWN_TOOL", "등록되지 않은 Tool입니다.")
    else:
        result = guard(tool_call["args"], request.runtime.context)

    if isinstance(result, dict):
        return ToolMessage(
            content=json.dumps(result, ensure_ascii=False),
            tool_call_id=tool_call["id"],
            name=tool_call["name"],
            status="error",
        )

    return handler(request)
```

### 실습: Context로 Tool 실행 허용하기

Context에 따라 아래 Tool의 실행을 허용하거나 차단해 보자.

허용된 경우에는 안내 문구를 조회하고, 허용되지 않은 경우에는 Tool을 실행하지 않고 거절 사유를 안내한다. 두 경우를 실행해 결과를 비교한다.

`agent.invoke(..., context={"allow_tool": True})`와 `False`인 경우를 각각 실행한다.


In [3]:
@tool
def get_notice() -> str:
    """안내 문구를 조회한다."""
    print("Tool 실행")
    return "오늘 수업은 오후 6시에 시작합니다."

@wrap_tool_call
def validate_tool_call(request: ToolCallRequest, handler):
    print("실행 전 검사:", request.tool_call["name"])
    result = request.runtime.context["allow_tool"]

    # 거절·취소·승인 대기이면 Tool 실행 없이 사유를 반환한다.
    if not result:
        return ToolMessage(
            content=json.dumps({"message" : "Tool 실행이 허용되지 않았습니다."}, ensure_ascii=False),
            tool_call_id=request.tool_call["id"],
            name=request.tool_call["name"],
            status="error",
        )

    print("Tool 실행:", request.tool_call["name"])
    return handler(request)

In [6]:
notice_agent = create_agent(
    model=llm,
    tools=[get_notice],
    system_prompt="Tool 결과를 바탕으로 한국어로 안내하세요.",
    middleware=[force_tool_call, validate_tool_call],
)

In [7]:
allowed_result = notice_agent.invoke(
    {"messages": [HumanMessage(content="오늘 안내 문구 알려줘.")]},
    context={"allow_tool": True},
)

for message in allowed_result["messages"]:
    message.pretty_print()

실행 전 검사: get_notice
Tool 실행: get_notice
Tool 실행
================================ Human Message =================================

오늘 안내 문구 알려줘.
================================== Ai Message ==================================

[]
Tool Calls:
  get_notice (call_1737419)
 Call ID: call_1737419
  Args:
================================= Tool Message =================================
Name: get_notice

오늘 수업은 오후 6시에 시작합니다.
================================== Ai Message ==================================

[{'type': 'text', 'text': '오늘 안내 문구는 다음과 같습니다.\n\n> **"오늘 수업은 오후 6시에 시작합니다."**', 'extras': {'signature': 'EvABCu0BAWkUfRP15F3yWnCTpV/XqV1/xyD+Db4YSwqVK25anC4JS/oV9vrU6DJiBJnFcnoRMdkobZEeqYgcg3aVDinZA7Q5qkgDrEaDNMGGObQW0crHiQMsCvGX9xzxhAbbrOOHzNGtINY+sSXhjE3jVziPMm5sINmdlSVjgOtsrdxyTdgKouRDQFqd+9abiq59BsK0CyTyxT6uKz/+R/wZojCG73038tIR0u/1q0wQnOZB5IZ6QCuG/wuD4uKbcMFg3Nf5AwPTXzApQ48WPEI1/E2qWzqw5hVJoifTeotvA3uKiRD0kdgXH2q09C3sgYR5'}}]


In [8]:
allowed_result = notice_agent.invoke(
    {"messages": [HumanMessage(content="오늘 안내 문구 알려줘.")]},
    context={"allow_tool": False},
)

for message in allowed_result["messages"]:
    message.pretty_print()

실행 전 검사: get_notice
================================ Human Message =================================

오늘 안내 문구 알려줘.
================================== Ai Message ==================================

[]
Tool Calls:
  get_notice (call_157208)
 Call ID: call_157208
  Args:
================================= Tool Message =================================
Name: get_notice

{"message": "Tool 실행이 허용되지 않았습니다."}
================================== Ai Message ==================================

[{'type': 'text', 'text': '죄송합니다. 현재 시스템 설정으로 인해 안내 문구를 조회할 수 없습니다. \n\n추후 기능이 정상 작동하거나 권한이 부여되면 다시 시도해 주시기 바랍니다.', 'extras': {'signature': 'Eo8ECowEAWkUfROw0jZnmsHDV9IihhfiNyUDHglYK4/x0520MIaSoPqGw/BFI65h6+c2+GtKd4UpdEMffUA+Pp85L7/gufkVROHE8IrPs718csSlSNXu4eA4KHdE8hkE2k2Yuju3G/2T5GF4u6MNUlDCpS4ezVVADL/zircAv25tnDP8jPmoT2eeVUaK1R6EIKiKXbGqE6RiFcxUxd/IodMVbk4UiE3dZQoiGFazBa2qj5EBUrKMmzaqEBvPdlSQlHhgMe4FGQYX9fwl82MNh+JizkTgRae1X06QoMCBLNeE1qJ4yqrtsLPcOfKmbFLZRUNyrGrxj0IgwhmLQjRnW5V5U9I5OaNA+QOWzk5blRaJsc/+tMbs